[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Which Driver


## What you will be able to do

Choose between the two drivers on evidence rather than on reputation. Run the same workload four
ways, synchronous psycopg as a labeled baseline, asyncpg one query at a time, and both drivers under
`asyncio.gather` through a pool, so that a number compares a driver with a driver rather than
serial work with concurrent work. Say which of the four rows in that table is the one that matters
for a service. And recognize the benchmark that reports asyncpg as hundreds of times faster because
nothing in it ran.


## The idea

### The problem

"asyncpg is faster" is true, false, or irrelevant depending on what you measure, and most
comparisons you will read measure the wrong thing: synchronous psycopg in a loop against asyncpg
under `asyncio.gather`. That compares one query at a time with eight at a time, and the answer is
about concurrency rather than about either driver.

psycopg has an asynchronous half and a pool. Once both drivers are asked to do the same thing, most
of the gap disappears, and what is left is small, specific and worth knowing.

### What actually differs

Three things. asyncpg decodes rows in its own binary protocol implementation rather than through
libpq, which shows up on wide results and nowhere else. psycopg is a DB-API driver with a
synchronous half, which asyncpg has no equivalent of. And asyncpg's API is smaller, stricter and
has per-call timeouts, where psycopg has a broader one and a lot more compatibility.

### Why measuring this is hard

The machine running this notebook talks to PostgreSQL over a Unix socket, which is the shortest
round trip that exists. Concurrency hides waiting, so a benchmark with no waiting in it understates
what concurrency buys. Every number here is therefore a floor, and the section with `pg_sleep` in it
is the one that stands in for a real network.

### Where this shows up

Choosing a driver at the start of a project, and defending the choice later. Also reading somebody
else's benchmark, which is a skill this notebook is mostly about.

### What this notebook covers

Four workloads, four ways each: a tiny query, a wide result, and the same again with latency added.
What each row says. The costs that do not show up in a timing at all. Then the missing `await` that
fakes a win, and the two errors it turns into.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio
import time

import asyncpg
import psycopg


def compared(baseline, measured):
    ratio = baseline / measured
    if ratio < 1.2:
        return "about the same"
    return "somewhat faster" if ratio < 3 else "several times faster"


async def main():
    print("asyncpg against psycopg, both one query at a time, no concurrency anywhere:")

    for label, sql, runs in (("one number, 500 times ", "SELECT count(*) FROM events", 500),
                             ("5000 rows, 20 times   ", "SELECT id, ts, kind, payload "
                                                        "FROM events", 20)):
        with psycopg.connect("dbname=guide") as sync:
            start = time.perf_counter()
            for _ in range(runs):
                sync.execute(sql).fetchall()
            blocking = time.perf_counter() - start

        conn = await asyncpg.connect(database="guide")
        start = time.perf_counter()
        for _ in range(runs):
            await conn.fetch(sql)
        waiting = time.perf_counter() - start
        await conn.close()

        print(f"  {label} {compared(blocking, waiting)}")


asyncio.run(main())
```

```
asyncpg against psycopg, both one query at a time, no concurrency anywhere:
  one number, 500 times  about the same
  5000 rows, 20 times    several times faster
```

Two workloads, no concurrency in either, and two different answers. On five hundred tiny queries the
two drivers are indistinguishable, which is not what the reputation says. On twenty wide results
asyncpg is several times faster, and that is a real difference with a real cause.


## Setup

Twelve imports, both drivers, both pools, the server, and the measuring apparatus.

- `psycopg`, `psycopg_pool` and `asyncpg`, which are the three things being compared
- `asyncio` runs the concurrent halves, `time` measures, and `warnings` and `gc` catch one warning
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

Three queries are defined once and used throughout: `SMALL` returns a number, `WIDE` returns five
thousand rows of four columns, and `SLOW` adds ten milliseconds of waiting to stand in for a
network. `blocking`, `asyncpg_serial`, `asyncpg_pooled` and `psycopg_pooled` are the four ways of
running one of them, `best_of` takes the fastest of three runs, and `compared` turns a ratio into a
band because a ratio on a shared machine is not repeatable to two decimals.


In [1]:
import asyncio
import gc
import getpass
import os
import subprocess
import sys
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

SMALL = "SELECT count(*) FROM events WHERE kind = 'click'"
WIDE = "SELECT id, ts, kind, payload FROM events"
SLOW = "SELECT pg_sleep(0.01), 1"                                   # ten milliseconds of waiting


def compared(baseline, measured):
    """A band, not a number: a ratio on a shared machine is not repeatable to two decimals."""
    ratio = baseline / measured
    if ratio < 1.2:
        return "about the same"
    return "somewhat faster" if ratio < 3 else "several times faster"


def best_of(run, repeats=3):
    """The fastest of several runs, which is the least noisy summary of a timing."""
    return min(run() for _ in range(repeats))


def blocking(sql, runs):
    """psycopg, one connection, one query at a time. This is the baseline everything is against."""
    def once():
        with psycopg.connect("dbname=guide", autocommit=True) as conn:
            start = time.perf_counter()
            for _ in range(runs):
                conn.execute(sql).fetchall()
            return time.perf_counter() - start
    return best_of(once)


async def best_of_async(run, repeats=3):
    return min([await run() for _ in range(repeats)])


async def asyncpg_serial(sql, runs):
    async def once():
        conn = await asyncpg.connect(database="guide")
        start = time.perf_counter()
        for _ in range(runs):
            await conn.fetch(sql)
        taken = time.perf_counter() - start
        await conn.close()
        return taken
    return await best_of_async(once)


async def asyncpg_pooled(sql, runs, size=8):
    async def once():
        pool = await asyncpg.create_pool(database="guide", min_size=size, max_size=size)
        start = time.perf_counter()
        await asyncio.gather(*(pool.fetch(sql) for _ in range(runs)))
        taken = time.perf_counter() - start
        await pool.close()
        return taken
    return await best_of_async(once)


async def psycopg_pooled(sql, runs, size=8):
    async def once():
        pool = psycopg_pool.AsyncConnectionPool("dbname=guide", min_size=size, max_size=size,
                                                open=False)
        await pool.open(wait=True, timeout=10)

        async def one():
            async with pool.connection() as conn:
                return await (await conn.execute(sql)).fetchall()

        start = time.perf_counter()
        await asyncio.gather(*(one() for _ in range(runs)))
        taken = time.perf_counter() - start
        await pool.close()
        return taken
    return await best_of_async(once)


print("server:", start_server())
print(report())
print("measuring against a local Unix socket, which is the least favorable place for a driver")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
measuring against a local Unix socket, which is the least favorable place for a driver


## Worked examples

### Row one: many small queries, one at a time

The cheapest possible query, run five hundred times, with nothing overlapping:


In [2]:
baseline = blocking(SMALL, 500)
print("psycopg, synchronous: the baseline")
print("asyncpg, one at a time:", compared(baseline, await asyncpg_serial(SMALL, 500)))


psycopg, synchronous: the baseline
asyncpg, one at a time: about the same


Nothing in it. Both drivers spend the whole time on round trips to a socket a few microseconds away,
and neither one's decoding matters because there is almost nothing to decode.

This row is the one most comparisons quietly leave out.

### Row two: the same queries, eight at a time

Now with a pool and `asyncio.gather`, which is the row every benchmark shows:


In [3]:
psycopg_time = await psycopg_pooled(SMALL, 500)
asyncpg_time = await asyncpg_pooled(SMALL, 500)

print("psycopg, synchronous: the baseline")
print("psycopg, async pool: ", compared(baseline, psycopg_time))
print("asyncpg, async pool: ", compared(baseline, asyncpg_time))
print()
print("and the two pools against each other:", compared(psycopg_time, asyncpg_time))


psycopg, synchronous: the baseline
psycopg, async pool:  somewhat faster
asyncpg, async pool:  several times faster

and the two pools against each other: somewhat faster


Both pools beat the baseline, and most of what they gained they gained for the same reason: eight
connections instead of one. That is why the last line is there. Measured against the synchronous
baseline, the asyncpg pool looks like the winner; measured against the other pool, which is the
comparison that holds concurrency constant, the remaining difference is much smaller.

Quoting only the asyncpg line against the synchronous baseline is the commonest way a benchmark
credits a driver with something the pool did.

### Row three: a wide result

Five thousand rows of four columns, one of them `jsonb`, twenty times:


In [4]:
baseline = blocking(WIDE, 20)
print("psycopg, synchronous: the baseline")
print("asyncpg, one at a time:", compared(baseline, await asyncpg_serial(WIDE, 20)))
print("asyncpg, async pool:  ", compared(baseline, await asyncpg_pooled(WIDE, 20, size=4)))


psycopg, synchronous: the baseline
asyncpg, one at a time: several times faster
asyncpg, async pool:   several times faster


Here is asyncpg's real advantage, and notice it appears without any concurrency at all. asyncpg
decodes the wire protocol itself, in Cython, straight into `Record` objects. psycopg goes through
libpq and builds tuples. On a hundred thousand values that difference adds up.

If your program reads large results, this row is your reason to choose asyncpg. If it reads a few
rows at a time, this row does not apply to you.

### Row four: the same workload with latency in it

Ten milliseconds of waiting per query, forty queries, which is what a database one network hop away
looks like:


In [5]:
baseline = blocking(SLOW, 40)
print("psycopg, synchronous: the baseline (40 x 10ms, one after another)")
print("psycopg, async pool:  ", compared(baseline, await psycopg_pooled(SLOW, 40)))
print("asyncpg, async pool:  ", compared(baseline, await asyncpg_pooled(SLOW, 40)))


psycopg, synchronous: the baseline (40 x 10ms, one after another)
psycopg, async pool:   several times faster
asyncpg, async pool:   several times faster


The largest gap in the notebook, and the two drivers are level. Forty waits of ten milliseconds
happen eight at a time instead of one at a time, and that is all this row measures.

It is also the row that most resembles production, and the one that says asynchronous code is worth
adopting. It says nothing at all about which driver to adopt it with.

### What does not show up in a timing

| Cost | psycopg | asyncpg |
|---|---|---|
| a synchronous option | yes, the same API | none at all |
| DB-API compatibility | yes, so libraries expect it | no |
| placeholders | `%s`, as everything else uses | `$1`, so queries are not portable |
| a `jsonb` column | a `dict` with no setup | a codec you register per connection |
| a write with no transaction block | provisional until commit | already committed |
| a per-call timeout | none, only `statement_timeout` | `timeout=` on every call |
| other databases | SQLite and MySQL have DB-API drivers too | PostgreSQL only |

Two of those rows are the ones that actually decide projects. A codebase with a synchronous half
cannot use asyncpg for it, and a team that has written `%s` everywhere is not porting for a decoding
speedup it may not be able to measure.

### When to reach for which

| Situation | Driver |
|---|---|
| a synchronous program | psycopg, and there is no alternative |
| a web service, either kind of workload | either, with a pool, and the pool is what matters |
| large result sets, read often | asyncpg |
| a codebase already using `%s` and DB-API | psycopg |
| you want one driver for PostgreSQL and SQLite | psycopg |
| you want per-query timeouts | asyncpg |
| you are not sure | psycopg |

The default is psycopg, because it does everything, its asynchronous half performs within noise of
asyncpg on every workload above except wide reads, and its placeholders and DB-API shape are what
the rest of the ecosystem expects. Choose asyncpg deliberately, for the decoding or for the API, and
not because a benchmark said it was faster.

### One workload, measured honestly, finished

Everything above as one function, because the shape of an honest comparison is itself the lesson:
name the baseline, run every contender on the same work, and report bands.


In [6]:
async def compare(sql, runs, label, size=8):
    """One workload, four ways, with synchronous psycopg named as the baseline."""
    baseline = blocking(sql, runs)
    return {
        "workload": label,
        "psycopg, synchronous": "the baseline",
        "asyncpg, serial": compared(baseline, await asyncpg_serial(sql, runs)),
        "psycopg, pooled": compared(baseline, await psycopg_pooled(sql, runs, size)),
        "asyncpg, pooled": compared(baseline, await asyncpg_pooled(sql, runs, size)),
    }


for result in [await compare(SMALL, 500, "500 tiny queries"),
               await compare(WIDE, 20, "20 wide results", size=4),
               await compare(SLOW, 40, "40 queries with 10ms of latency")]:
    print(result.pop("workload"))
    for way, verdict in result.items():
        print(f"  {way:22} {verdict}")


500 tiny queries
  psycopg, synchronous   the baseline
  asyncpg, serial        about the same
  psycopg, pooled        somewhat faster
  asyncpg, pooled        several times faster
20 wide results
  psycopg, synchronous   the baseline
  asyncpg, serial        several times faster
  psycopg, pooled        about the same
  asyncpg, pooled        several times faster
40 queries with 10ms of latency
  psycopg, synchronous   the baseline
  asyncpg, serial        about the same
  psycopg, pooled        several times faster
  asyncpg, pooled        several times faster


Read down the columns rather than across, and the three workloads say three different things.

On the tiny queries the serial row is flat and both pooled rows move, so the pool is doing the work.
On the queries with latency the same thing happens and more strongly, which is the same lesson.

The wide result is the interesting one: the asyncpg serial row moves without any concurrency at all,
and psycopg's pool gains little or nothing over psycopg alone. Decoding five thousand rows is work
your own process has to do, and concurrency cannot hide work that is not waiting. That is the one
place in this notebook where the driver itself is the answer.

Anything reporting a single number for "asyncpg versus psycopg" would have had to throw all of that
away.

### Where each part came from

| In the comparison | What it relies on | The section that showed it |
|---|---|---|
| synchronous psycopg as the baseline | the thing everything else is measured against | Row one |
| `psycopg_pool.AsyncConnectionPool` | psycopg having a concurrent half at all | **Connection Pools** |
| `asyncio.gather` over a pool | one connection per concurrent query | **AsyncConnection** |
| `pool.fetch` with no `acquire` | asyncpg borrowing and returning in one call | **Connection Pools** |
| `best_of` and `compared` | a band rather than a number | **Pipeline Mode** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/16-which-driver-solutions.ipynb).

**1.** Time the same small query on both drivers with no concurrency, and print the band.


In [7]:
# your code here


**2.** Time a wide result on both drivers with no concurrency.


In [8]:
# your code here


**3.** Run the small query five hundred times through each driver's pool and compare the two pools
with each other rather than with the baseline.


In [9]:
# your code here


**4.** Add ten milliseconds of latency to each query and show what concurrency buys.


In [10]:
# your code here


**5.** Show that a pool of one removes the whole advantage.


In [11]:
# your code here


**6.** Count the connections each way opens, using `pg_stat_database.sessions`.


In [12]:
# your code here


## Common errors

### RuntimeWarning: coroutine 'Connection.fetch' was never awaited


In [13]:
conn = await asyncpg.connect(database="guide")

with warnings.catch_warnings(record=True) as caught:                # caught, not printed loose
    warnings.simplefilter("always")

    start = time.perf_counter()
    for _ in range(200):
        conn.fetch(SMALL)                                           # the await is missing
    without = time.perf_counter() - start

    start = time.perf_counter()
    for _ in range(200):
        await conn.fetch(SMALL)
    with_it = time.perf_counter() - start
    gc.collect()

ratio = with_it / without
size = "tens" if ratio < 100 else "hundreds" if ratio < 1000 else "thousands"
print(f"the benchmark would report {size} of times faster")
print("Python said, 200 times over:", caught[0].message)


the benchmark would report hundreds of times faster
Python said, 200 times over: coroutine 'Connection.fetch' was never awaited


Every one of those two hundred calls created a coroutine and threw it away. Nothing was sent, nothing
was decoded, nothing was waited for, and the loop measured the cost of constructing an object.

This is the single most common way an asynchronous benchmark is wrong, and it is worth suspecting
any result in the hundreds. The warnings are the evidence, and they usually scroll past unread.

### TypeError: 'coroutine' object is not subscriptable


In [14]:
rows = conn.fetch(WIDE)                                             # still no await
print(rows[0])


TypeError: 'coroutine' object is not subscriptable

The same mistake, one line further on, and this one at least fails. A coroutine is not a list, so the
moment the benchmark tries to use a result rather than discard it, Python says so.

`len(rows)` gives `object of type 'coroutine' has no len()`, for the same reason. The habit that
prevents all of it is to `await` at the point of the call, and the first line below is the other
half of the habit: a coroutine you have decided not to run is closed rather than dropped, which is
what stops the `RuntimeWarning` arriving somewhere else later.


In [15]:
rows.close()                                                        # a coroutine you decide not to
gc.collect()                                                        # run should be closed, not lost

rows = await conn.fetch(WIDE)
print("with the await:", type(rows).__name__, "of", len(rows), "|", rows[0]["kind"])


with the await: list of 5000 | view


### No error: one number for two different questions


In [16]:
baseline = blocking(SMALL, 500)
serial = compared(baseline, await asyncpg_serial(SMALL, 500))
pooled = compared(baseline, await asyncpg_pooled(SMALL, 500))

print("asyncpg against synchronous psycopg, on identical work:", serial)
print("asyncpg pooled against synchronous psycopg:            ", pooled)
print()
print("both are true, and only one of them is about the driver")


asyncpg against synchronous psycopg, on identical work: about the same
asyncpg pooled against synchronous psycopg:             several times faster

both are true, and only one of them is about the driver


Two honest measurements of the same driver on the same query, and they disagree because they are
answers to different questions. The second one is what gets quoted, and the concurrency in it
belongs to `asyncio.gather` and the pool rather than to asyncpg.

The test to apply to any comparison, including this notebook's: is the baseline doing the same
amount of work at the same time as the contender? If it is not, the number measures the difference
in the harness.


In [17]:
await conn.close()
print("connection closed")


connection closed


## Recap

- Compare drivers on identical work. Synchronous psycopg in a loop against asyncpg under
  `asyncio.gather` measures concurrency, not drivers.
- On many small queries the two are indistinguishable, serially and pooled.
- On wide results asyncpg is several times faster, with no concurrency involved, because it decodes
  the protocol itself rather than through libpq.
- With real latency, concurrency is worth several times the serial baseline, and both drivers get
  the same benefit. That row argues for a pool, not for a driver.
- Measure over something slower than a Unix socket if you can. Every number measured here is a floor.
- The costs that never appear in a timing decide most projects: a synchronous half, DB-API,
  `%s` against `$1`, `jsonb` decoding, and what a write does without a transaction block.
- A missing `await` reports hundreds of times faster and runs nothing. Suspect any result that large,
  and read the `RuntimeWarning`s.


## What is next

**An Event Store** is the guide's last notebook and uses both drivers on purpose: psycopg for the
`COPY` load and the pipelined writes, asyncpg for the concurrent readers and the listener, with one
line saying why each side got the one it got.


---

&#8592; **Previous:** [LISTEN and NOTIFY](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/15-listen-and-notify.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
